In [29]:
import trimesh
# JUPYTER CELL 2 — Paramètres
from pathlib import Path
import numpy as np

# Fichier GLB/GLTF source
GLTF_PATH = Path("../client/public/models/city2.glb")      # <-- adapte ce chemin

# Résolution de la heightmap
N = 1000                                  # 256..1024 selon ta scène et le temps dispo

# Domaine XZ à baker (auto=True => utilise le bounding box du modèle)
AUTO_BOUNDS = True
MIN_X, MAX_X = -50.0, 50.0               # ignorés si AUTO_BOUNDS=True
MIN_Z, MAX_Z = -50.0, 50.0

SCALE_X=5
SCALE_Y=4
SCALE_Z=5

# Altitude de départ du rayon et bornes de quantification (visuel/jeu)
Y_MIN, Y_MAX = 0.0, 50.0                  # clamp & quantize dans [Y_MIN, Y_MAX]

# Sorties
OUT_BIN  = Path("../server/public/heightmap.bin")
OUT_META = Path("../server/public/heightmap.meta.json")

In [31]:
scene = trimesh.load(gltf_path.as_posix(), force='scene').scaled((SCALE_X,SCALE_Y,SCALE_Z))  # Scene
len([len(t) for t in scene.triangles])

15633

In [32]:
data = np.zeros(shape=(N,N),dtype=np.uint16)

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint16)

In [43]:
import scipy

for t in scene.triangles:
    t_minx = max(min(min([v[0] for v in t]),MAX_X),MIN_X)
    t_maxx = max(min(max([v[0] for v in t]),MAX_X),MIN_X)
    t_minz = max(min(min([v[2] for v in t]),MAX_Z),MIN_Z)
    t_maxz = max(min(max([v[2] for v in t]),MAX_Z),MIN_Z)

    i_minx = int(np.ceil((N-1)*(t_minx-MIN_X)/(MAX_X-MIN_X)))
    i_maxx = int(np.floor((N-1)*(t_maxx-MIN_X)/(MAX_X-MIN_X)))
    i_minz = int(np.ceil((N-1)*(t_minx-MIN_Z)/(MAX_Z-MIN_Z)))
    i_maxz = int(np.floor((N-1)*(t_maxx-MIN_Z)/(MAX_Z-MIN_Z)))

    A = [[t[0][0]-t[2][0],t[1][0]-t[2][0]],
        [t[0][2]-t[2][2],t[1][2]-t[2][2]]]

    if(np.linalg.det(A)!=0):
        LU = scipy.linalg.lu_factor(A)
    
        for ix in range(i_minx,i_maxx+1):
            for iz in range(i_minz,i_maxz+1):
                x = ix*(MAX_X-MIN_X)/(N-1)+MIN_X
                z = iz*(MAX_Z-MIN_Z)/(N-1)+MIN_Z
                b = [x-t[2][0],z-t[2][2]]
                g = scipy.linalg.lu_solve(LU,b)
                if(0<=g[0]<=1 and 0<=g[1]<=1 and g[0]+g[1]<=1): #if point is in triangle
                    y = g[0]*t[0][1]+g[1]*t[1][1]+(1-g[0]-g[1])*t[2][1]
                    y = max(min(y,Y_MAX),Y_MIN)
                    y_val = int((2**16-1)*(y-Y_MIN)/(Y_MAX-Y_MIN))
                    data[ix][iz] = max( data[ix][iz],y_val)
                    
            
    

In [48]:
data[500][400]

0

In [50]:
import json

OUT_BIN.write_bytes(data.tobytes(order='C'))
OUT_META.write_text(json.dumps({
    "N": int(N),
    "minX": float(MIN_X), "maxX": float(MAX_X),
    "minZ": float(MIN_Z), "maxZ": float(MAX_Z),
    "ymin": float(Y_MIN), "ymax": float(Y_MAX),
    "scaleX":float(SCALE_X),"scaleY":float(SCALE_Y),"scaleZ":float(SCALE_Z),
}, indent=2))

163